In [ ]:
#写一个大循环，循环不同的分辨率
# 匹配数目与平均降水
from scipy import stats
from scipy.stats import gaussian_kde
import numpy as np
import matplotlib as mpl
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeat
import matplotlib.pyplot as plt
from cartopy.io.shapereader import Reader
import matplotlib.ticker as mticker
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.feature import ShapelyFeature
from matplotlib.colorbar import ColorbarBase
from matplotlib.colors import BoundaryNorm
a=0.2
p = int(7 / a)
q = int(11 / a)

In [ ]:
#对数据格点化
guage_pre_all = np.load(r"guage_pre_all.npy")
df_posi=pd.read_csv('df_posi_02—10.csv')
gauge_mean=np.nanmean(guage_pre_all,axis=0)
zero_indices =np.array( np.where(gauge_mean == 0)[0] )
df_posi = df_posi.drop(index=zero_indices)

In [ ]:
guage_pre_all.shape

In [ ]:


# gauge_pre_grided=np.zeros((43848,p,q))
# gauge_pre_num=np.zeros((43848,p,q))
# for t in range(43848):
#     print(t)
#     for i in range(len(df_posi["lons"])):
#         if i in zero_indices:
#             continue
#         x=int((df_posi["lons"][i]-111)/a)
#         y=int((df_posi["lats"][i]-30)/a)
#         if guage_pre_all[t,i]!=guage_pre_all[t,i]:
#             continue
#         gauge_pre_grided[t,y,x]=gauge_pre_grided[t,y,x]+guage_pre_all[t,i]
#         gauge_pre_num[t,y,x] = gauge_pre_num[t,y,x]+1
# gauge_pre_grided=gauge_pre_grided/gauge_pre_num

In [ ]:
'''
5月:31天 744

6月:30天
7月:31天
8月:31天 2952

9月:30天
# 10月:31天
11月:30天 5136

12月:31天
1月:31天
2月:29天 7320

3月:31天
4月:30天 8784
'''
# gauge_pre_grided.shape

In [ ]:
# gauge_pre_num

In [ ]:
# np.save("gauge_pre_num.npy",gauge_pre_num)

In [ ]:
# np.save("gauge_pre_grided.npy",gauge_pre_grided)

In [ ]:
# gauge_pre_num=np.load("gauge_pre_num.npy")

In [ ]:
gauge_pre_grided=np.load("gauge_pre_grided.npy")

In [ ]:
gauge_pre_grided.shape

In [ ]:
np.nansum(np.nanmean(np.append(gauge_pre_grided[0:3672],gauge_pre_grided[-721:-1],axis=0),axis=(1,2)))

In [ ]:
gauge_pre_grided[5136:7320].shape

In [ ]:
np.nansum(np.nanmean(gauge_pre_grided[5136:7320],axis=(1,2)))/np.nansum(np.nanmean(gauge_pre_grided,axis=(1,2)))

In [ ]:
sate_pre_all = np.load(r"precip_all_hour_satellite.npy")
#把之前读取数据有的错订正
# sate_pre_all=np.transpose(sate_pre_all, (0, 2, 1))
# sate_pre_all = np.concatenate((sate_pre_all[61*24:], sate_pre_all[:61*24]), axis=0)

In [ ]:
# np.save('precip_all_hour_satellite.npy',sate_pre_all)

In [ ]:


huatus = np.zeros((8,) + gauge_pre_grided.shape[1:])  # 前4个是地面数据，后4个是卫星数据

# 定义每年的小时数（已知两个闰年）
hours_per_year = [8784, 8760, 8760, 8760, 8784]

# 预计算各季节起始索引（基于每年1月1日0时为起点）
season_slices = [
    # 春季：3-5月（按闰年计算，实际使用时根据年份索引选择）
    {"start": 31*24 + 28*24, "end": 31*24 + 28*24 + 92*24},  # 3/1 0:00 ~ 5/31 23:00
    
    # 夏季：6-8月
    {"start": 31*24 + 28*24 + 92*24, "end": 31*24 + 28*24 + 92*24 + 92*24},
    
    # 秋季：9-11月
    {"start": 31*24 + 28*24 + 184*24, "end": 31*24 + 28*24 + 184*24 + 91*24},
    
    # 冬季：当年12月 + 次年1-2月（修正为仅当前年）
    {"start": 334*24, "end": 365*24},  # 12/1 0:00 ~ 12/31 23:00
    {"pre_start": 0, "pre_end": 59*24}  # 1/1 0:00 ~ 2/28 23:00
]

# 动态生成各年实际索引（考虑闰年）
def get_season_indices(year_idx, is_leap):
    base = sum(hours_per_year[:year_idx])
    leap_offset = 24 if is_leap else 0
    
    return {
        "spring": (
            base + (31*24 + 28*24 + leap_offset),  # 3/1 0:00
            base + (31*24 + 28*24 + 92*24 + leap_offset)
        ),
        "summer": (
            base + (31*24 + 28*24 + 92*24 + leap_offset),
            base + (31*24 + 28*24 + 184*24 + leap_offset)
        ),
        "autumn": (
            base + (31*24 + 28*24 + 184*24 + leap_offset),
            base + (31*24 + 28*24 + 275*24 + leap_offset)
        ),
        "winter": [
            (base + 334*24, base + (366 if is_leap else 365)*24),  # 12/1 ~ 年底
            (base, base + 59*24 + (24 if is_leap else 0))          # 1/1 ~ 2/29/28
        ]
    }


# 初始化季节数据容器
gauge_seasons = [[] for _ in range(4)]
sate_seasons = [[] for _ in range(4)]

# 处理每个年份的数据
cursor = 0
for year in range(5):
    is_leap = (hours_per_year[year] == 8784)
    indices = get_season_indices(year, is_leap)
    
    # 切割四季数据
    for i, season in enumerate(["spring", "summer", "autumn"]):
        start, end = indices[season]
        gauge_seasons[i].append(gauge_pre_grided[start:end])
        sate_seasons[i].append(sate_pre_all[start:end])
    
    # 冬季特殊处理：12月 + 1-2月（同一年内）
    winter12 = gauge_pre_grided[indices["winter"][0][0]:indices["winter"][0][1]]
    winter1_2 = gauge_pre_grided[indices["winter"][1][0]:indices["winter"][1][1]]
    gauge_seasons[3].append(np.concatenate([winter1_2, winter12]))
    
    sate_winter12 = sate_pre_all[indices["winter"][0][0]:indices["winter"][0][1]]
    sate_winter1_2 = sate_pre_all[indices["winter"][1][0]:indices["winter"][1][1]]
    sate_seasons[3].append(np.concatenate([sate_winter1_2, sate_winter12]))

# 计算平均值
for i in range(4):
    huatus[i] = np.nanmean(np.concatenate(gauge_seasons[i], axis=0), axis=0)
    huatus[i+4] = np.nanmean(np.concatenate(sate_seasons[i], axis=0), axis=0)

# 维度调整（若卫星数据存储顺序不同）
# huatus[4:] = np.transpose(huatus[4:], (0, 2, 1))  # 假设需要(lat, lon) -> (lon, lat)

print("地面数据四季平均形状:", huatus[:4].shape)
print("卫星数据四季平均形状:", huatus[4:].shape)

In [ ]:
np.array(gauge_seasons[1]).shape

In [ ]:
np.nanmean(np.nansum(np.append(gauge_pre_grided[0:744],gauge_pre_grided[7320:8784],axis=0),axis=0))

In [ ]:
# np.save('gauge_pre_grided.npy',gauge_pre_grided)

In [ ]:
# gauge_pre_grided=np.load('gauge_pre_grided.npy')

In [ ]:
gauge_pre_grided.shape

In [ ]:
# pre_after_above = np.load(r"H:\bishetestresult\pre_imerg_above.npy")
# pre_after_down =np.load(r"H:\bishetestresult\pre_day_down.npy")
# # 插值后的数据相减，平方，平均，再开方,绝对误差
# pre_after_above=np.where(pre_after_above<1,1,pre_after_above)
# pre_after_down=np.where(pre_after_down<1,1,pre_after_down)

# pre = (pre_after_down - pre_after_above) ** 2
# pre_y = np.nanmean(pre, axis=(0,2))  
# ab_error = np.sqrt(pre_y)

# # 相对误差，相减再除以地基数据，求绝对值，求平均
# pre_after_down_nan = np.where(pre_after_down < 1, 1, pre_after_down)
# error = abs(pre_after_down - pre_after_above) / pre_after_down_nan  # 五维
# error_y = np.nanmean(error, axis=(0,2))  # 五维数组，对第三维小时求平均
# re_error = error_y*100

# #bias
# bias=np.nanmean(pre_after_above - pre_after_down,axis=(0,2))

In [ ]:
# huatus={}
# huatus[0]=np.nanmean(ab_error[2:5],axis=0)#三四五月份的降水
# huatus[1]=np.nanmean(ab_error[5:8],axis=0)
# huatus[2]=np.nanmean(ab_error[8:11],axis=0)
# huatus[3]=np.nanmean(ab_error[[0,1,11]],axis=0)

# huatus[4]=np.nanmean(re_error[2:5],axis=0)#三四五月份的降水
# huatus[5]=np.nanmean(re_error[5:8],axis=0)
# huatus[6]=np.nanmean(re_error[8:11],axis=0)
# huatus[7]=np.nanmean(re_error[[0,1,11]],axis=0)

# huatus[8]=np.nanmean(bias[2:5],axis=0)#三四五月份的降水
# huatus[9]=np.nanmean(bias[5:8],axis=0)
# huatus[10]=np.nanmean(bias[8:11],axis=0)
# huatus[11]=np.nanmean(bias[[0,1,11]],axis=0)

In [ ]:
plt.rcParams["font.sans-serif"] = ["Arial"]  # 用于显示中文#Arial
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.unicode_minus"] = False  # 用于显示中文


# --设置shp路径，数据集已公开
shp_path = r"E:\0000000000\map_data\bou2_4p.dbf"
# --设置tif路径，数据集已公开
tif_path = r"E:\0000000000\map_data\地形数据\NE1_50M_SR_W.tif"

bins = np.arange(0,4.01,0.01)
#bins = np.arange(0,0.2,0.01)
nbin = len(bins) + 1
cmap = mpl.cm.get_cmap("rainbow", nbin)
norm = mpl.colors.BoundaryNorm(bins, nbin,extend='both')
N = cmap.N

In [ ]:
np.nansum(huatus[3])/(np.nansum(huatus[0])+np.nansum(huatus[2])+np.nansum(huatus[1])+np.nansum(huatus[3]))

In [ ]:
i=3
aaa=huatus[i].flatten()
bbb=huatus[i+4].flatten()
mask = ~(np.isnan(aaa) | np.isnan(bbb))
np.corrcoef(aaa[mask], bbb[mask])

In [ ]:
sate_pre_all.shape

In [ ]:
gauge_pre_grided.shape

In [ ]:
huatus[4]=huatus[4]+huatus[0]-huatus[0]
huatus[5]=huatus[5]+huatus[1]-huatus[1]
huatus[6]=huatus[6]+huatus[2]-huatus[2]
huatus[7]=huatus[7]+huatus[3]-huatus[3]

In [ ]:
i=0
aaa=huatus[i].flatten()
bbb=huatus[i+4].flatten()
mask = ~(np.isnan(aaa) | np.isnan(bbb))
np.corrcoef(aaa[mask], bbb[mask])

In [ ]:

fig, axs = plt.subplots(2, 4, figsize=(12, 4), dpi=300,subplot_kw={'projection': ccrs.PlateCarree()}#,gridspec_kw={'hspace': 0.05 }#'wspace': 0.0,
                        )
axs=axs.ravel()
for i in range(8):
    ax = axs[i]
    ax.coastlines()
    ax.patch.set_facecolor("lightgrey") 
    provinces = ShapelyFeature(Reader(shp_path).geometries(), ccrs.PlateCarree(), edgecolor="k", facecolor="none")
    ax.add_feature(provinces, lw=0.6, zorder=2)
    
    ax.set_extent([111, 122, 30, 37], crs=ccrs.PlateCarree())
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.8, color="gray", linestyle=":")
    gl.top_labels, gl.bottom_labels, gl.right_labels, gl.left_labels = False, False, False, False
    gl.xlocator = mticker.FixedLocator(np.arange(111, 122, 0.5))
    gl.ylocator = mticker.FixedLocator(np.arange(30, 37, 0.5))
    
    if i==0 or i==4  :
        ax.set_yticks(np.arange(30, 38, 2), crs=ccrs.PlateCarree())
        ax.yaxis.set_major_formatter(LatitudeFormatter())
        ax.tick_params(labelcolor="k", length=2, labelsize=10,rotation=0,pad=2)

    if i in[4,5,6,7]  :
        ax.set_xticks(np.arange(111, 123, 3), crs=ccrs.PlateCarree())
        ax.xaxis.set_major_formatter(LongitudeFormatter())
        ax.tick_params(labelcolor="k", length=2, labelsize=10,rotation=0,pad=2)

    im = ax.imshow(
        huatus[i]*10,
        #alpha=colors[i,j],
        cmap=cmap,
        norm=norm,
        origin="lower",
        extent=(111, 122, 30, 37),
    )



#plt.tight_layout()
axs[0].set_title('(a) Spring Gauges Rain',fontsize=10,loc='left')
axs[1].set_title('(b) Summer Gauges Rain',fontsize=10,loc='left')
axs[2].set_title('(c) Fall Gauges Rain',fontsize=10,loc='left')
axs[3].set_title('(d) Winter Gauges Rain',fontsize=10,loc='left')
axs[4].set_title('(e) Spring IMERG Rain',fontsize=10,loc='left')
axs[5].set_title('(f) Summer IMERG Rain',fontsize=10,loc='left')
axs[6].set_title('(g) Fall IMERG Rain',fontsize=10,loc='left')
axs[7].set_title('(h) Winter IMERG Rain',fontsize=10,loc='left')

# axs[0].annotate('(a)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[1].annotate('(b)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[2].annotate('(c)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[3].annotate('(d)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[4].annotate('(e)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[5].annotate('(f)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[6].annotate('(g)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)
# axs[7].annotate('(h)', xy=(0.97, 0.8), xycoords='axes fraction', fontsize=12, ha='right', va='top', bbox=dict(boxstyle="square,pad=0.2", fc="white", ec="none"),alpha=0.8)

# axs[0].set_ylabel('rain_gauge',fontsize=12)
# axs[4].set_ylabel('rain_IMERG',fontsize=12)

cbar_ax = fig.add_axes([0.2, 0.01, 0.6, 0.04])
cbar=plt.colorbar (im,cax=cbar_ax,extend='max',orientation='horizontal',ticks=np.arange(0,5.5,0.5))

font = {#'family' : 'serif',
                #'color'  : 'darkred',
                #'weight' : 'normal',
                'size'   : 12,}
cbar.set_label('precipitation (0.1 mm h$^{-1}$)',fontdict=font)
cbar.ax.tick_params(labelsize=13)
cbar.minorticks_off()

plt.show()